# 02 — EfficientNet-B0 Baseline Benchmark (Member 2)

This notebook implements the **Person 2 (Member 2)** assigned approach: **EfficientNet-B0** transfer learning on the small, imbalanced four-class lettuce nutritional deficiency dataset (`FN`, `-N`, `-P`, `-K`).

### Benchmark Protocol Requirements:
- **Architecture:** Pretrained EfficientNet-B0 (`efficientnet_b0.ra_in1k` via `timm`) with a replaced 4-class linear classification head.
- **Evaluation Protocol:** Repeated Stratified 4-Fold Cross-Validation across 3 random seeds (seeds 42, 123, 2026 = 12 total runs).
- **Shared Split Manifests:** Consumes `data/splits/seed_{seed}/fold_{fold}.csv` generated by the integration coordinator (Person 1). No splits are independently regenerated.
- **Watermark-Safe Preprocessing:** Preprocessing crops the 1024×1024 source images to 960×960 top-centered (`watermark_safe_crop`) to remove class-correlated DALL-E color bar artifacts as documented in `docs/dataset_audit.md`.
- **Training Schedule:** 15 epochs max, AdamW optimizer, class-weighted cross-entropy loss computed from training partition only. Epochs 1–3 train the head with frozen backbone and fixed BatchNorm statistics; epochs 4–15 fine-tune all parameters.
- **Model Selection:** Validation macro F1 (ties broken by lower validation loss).
- **Deliverables:** Recomputable prediction CSV, seed and across-seed metrics, confusion matrices, training curves, efficiency profiling, and error analysis.

In [ ]:
from pathlib import Path
import sys
import torch
import pandas as pd
import numpy as np

# Set repository root
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

IMAGE_ROOT = PROJECT_ROOT / "data" / "raw"
SPLITS_DIR = PROJECT_ROOT / "data" / "splits"
OUTPUT_DIR = PROJECT_ROOT / "results" / "member_2"

print(f"Project root: {PROJECT_ROOT}")
print(f"Image root:   {IMAGE_ROOT}")
print(f"Splits dir:   {SPLITS_DIR}")
print(f"PyTorch:      {torch.__version__} (CUDA: {torch.cuda.is_available()})")

## 1. Inspect Shared Split Manifests

We verify that all 12 split manifests exist and check the partition counts. Each run has ~121 training images, 31 validation images, and ~51 held-out test images.

In [ ]:
manifest_paths = sorted(SPLITS_DIR.glob("seed_*/fold_*.csv"))
assert len(manifest_paths) == 12, f"Expected 12 split files, found {len(manifest_paths)}"

summary = []
for path in manifest_paths:
    df = pd.read_csv(path)
    summary.append({
        "seed": path.parent.name,
        "fold": path.stem,
        "total": len(df),
        "train": (df["partition"] == "train").sum(),
        "validation": (df["partition"] == "validation").sum(),
        "test": (df["partition"] == "test").sum(),
        "train_FN": len(df[(df["partition"] == "train") & (df["label"] == "FN")]),
        "val_FN": len(df[(df["partition"] == "validation") & (df["label"] == "FN")]),
        "test_FN": len(df[(df["partition"] == "test") & (df["label"] == "FN")]),
    })
pd.DataFrame(summary)

## 2. Load Configuration and Inspect EfficientNet-B0 Architecture

We inspect the EfficientNet-B0 model from `timm` with ImageNet-1k pretrained weights.

In [ ]:
import timm
from lettuce_benchmark.config import load_config
from models.efficientnet_b0.train import create_model

config = load_config(PROJECT_ROOT / "models" / "efficientnet_b0" / "config.yaml")
print("Configuration loaded:")
print(json.dumps(config, indent=2))

model = create_model(config, weights=None)
total_params = sum(p.numel() for p in model.parameters())
classifier_params = sum(p.numel() for p in model.classifier.parameters())
backbone_params = total_params - classifier_params

print(f"\nEfficientNet-B0 Total Parameters:      {total_params:,}")
print(f"Backbone Parameters:                  {backbone_params:,}")
print(f"Classifier Head (4-class) Parameters: {classifier_params:,}")
print(f"Classifier Head structure:            {model.classifier}")

## 3. Run Benchmark Pipeline across All 12 Shared Runs

We execute `models.efficientnet_b0.train` across the 3 seeds and 4 folds.

In [ ]:
# Execute the benchmark runner
!python -m models.efficientnet_b0.train --config models/efficientnet_b0/config.yaml --output-dir outputs/efficientnet_b0

## 4. Recompute Metrics and Generate Deliverables

Using `models.efficientnet_b0.summarize`, the evaluator recomputes official out-of-fold metrics per seed and aggregates across seeds, creating `results/member_2/`.

In [ ]:
!python -m models.efficientnet_b0.summarize --runs-dir outputs/efficientnet_b0 --output-dir results/member_2

## 5. Review Results and Metrics

We load and display the final recomputed benchmark metrics, confusion matrices, and efficiency measurements.

In [ ]:
metrics_df = pd.read_csv(OUTPUT_DIR / "metrics.csv")
print("\n--- Benchmark Metrics Summary ---")
display(metrics_df)

efficiency_file = OUTPUT_DIR / "efficiency.json"
if efficiency_file.is_file():
    print("\n--- Efficiency Profile ---")
    with efficiency_file.open() as f:
        print(f.read())

## 6. Display Visualizations

Display training curves and out-of-fold confusion matrices for all 3 seeds.

In [ ]:
from IPython.display import Image as IPImage, display as ip_display

curves_img = OUTPUT_DIR / "training_curves.png"
if curves_img.is_file():
    print("Training Curves:")
    ip_display(IPImage(filename=str(curves_img)))

cm_img = OUTPUT_DIR / "confusion_matrix.png"
if cm_img.is_file():
    print("Confusion Matrices:")
    ip_display(IPImage(filename=str(cm_img)))